# Quantum Funding Chart — Smooth World Edition (v3)

Parser fix: the world SVG commands and numeric coordinates are tokenized separately. If this notebook was already open in Colab, reload the page before running it.

**Run all cells from the top after Runtime → Restart session.**

In [ ]:
from pathlib import Path
import os, subprocess

PROJECT_DIR = Path("/content/mit-prox-rd")
REPO_URL = "https://github.com/qimono-repos/mit-prox-rd.git"

if not (PROJECT_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--branch", "trunk", "--single-branch", REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin", "trunk"], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", "-B", "trunk", "FETCH_HEAD"], check=True)

os.chdir(PROJECT_DIR)
print("Ready:", Path.cwd())

## 1. Imports and Data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, PathPatch
from matplotlib.path import Path as MplPath
import numpy as np
from pathlib import Path

df = pd.read_csv("assets/quantum_funding_updated.csv", encoding="utf-8-sig")
print(f"Loaded {len(df)} records.")
display(df.head())

## 2. Configuration

In [ ]:
values = {
    "United States": 1006, "Canada": 38.5, "Brazil": 5.5,
    "United Kingdom": 330, "Germany": 848, "Netherlands": 85,
    "Switzerland": 23, "Austria": 24, "Italy": 36,
    "India": 90, "South Korea": 140, "Japan": 241,
    "Singapore": 44, "Australia": None, "China": 15000,
}

pos = {
    "Canada": (-105, 57), "United States": (-99, 37), "Brazil": (-52, -10),
    "United Kingdom": (-3, 54), "Netherlands": (5, 53), "Germany": (11, 51),
    "Switzerland": (8, 47), "Austria": (14, 47), "Italy": (12, 42),
    "India": (79, 22), "South Korea": (128, 36), "Japan": (139, 37),
    "Singapore": (104, 1.3), "Australia": (134, -25), "China": (105, 35),
}

## 3. Robust SVG World Layer

In [ ]:
from matplotlib.patches import PathPatch
from matplotlib.path import Path as MplPath
import xml.etree.ElementTree as ET
import re

def parse_svg_path(d):
    """Parse the M/C/Z commands used by this project's world silhouette SVG.
    SVG path commands are tokenized separately from numeric coordinates, so a
    command letter can never accidentally be passed to float().
    """
    token_pattern = re.compile(r"[MmCcZz]|[-+]?(?:\d*\.\d+|\d+\.?\d*)(?:[eE][-+]?\d+)?")
    tokens = token_pattern.findall(d.replace(",", " "))
    vertices, codes = [], []
    i = 0
    command = None
    current = (0.0, 0.0)
    subpath_start = (0.0, 0.0)

    def read_pair(index, relative=False):
        if index + 1 >= len(tokens) or tokens[index] in "MmCcZz" or tokens[index + 1] in "MmCcZz":
            raise ValueError(f"Expected x,y coordinates in SVG path near {tokens[index:index+5]}")
        x, y = float(tokens[index]), float(tokens[index + 1])
        if relative:
            x += current[0]
            y += current[1]
        return (x, y)

    while i < len(tokens):
        if tokens[i] in ("M", "m", "C", "c", "Z", "z"):
            command = tokens[i]
            i += 1
            if command in ("Z", "z"):
                vertices.append(subpath_start)
                codes.append(MplPath.CLOSEPOLY)
                current = subpath_start
                command = None
                continue
        if command is None:
            raise ValueError(f"SVG path data has coordinates without a command near {tokens[i:i+5]}")

        if command in ("M", "m"):
            point = read_pair(i, relative=(command == "m"))
            vertices.append(point)
            codes.append(MplPath.MOVETO)
            current = subpath_start = point
            i += 2
            # SVG treats subsequent coordinate pairs after M/m as implicit L/l.
            # This project's world paths use only one pair after M/m; reject extras
            # rather than silently misread an unsupported path.
            if i < len(tokens) and tokens[i] not in ("M", "m", "C", "c", "Z", "z"):
                raise ValueError("World SVG uses an unsupported implicit line after M; use explicit C or Z commands.")
        elif command in ("C", "c"):
            if i + 5 >= len(tokens):
                raise ValueError(f"Incomplete cubic curve in SVG path near {tokens[i:]}")
            relative = command == "c"
            p1 = read_pair(i, relative)
            p2 = read_pair(i + 2, relative)
            p3 = read_pair(i + 4, relative)
            vertices.extend((p1, p2, p3))
            codes.extend((MplPath.CURVE4, MplPath.CURVE4, MplPath.CURVE4))
            current = p3
            i += 6
        else:
            raise ValueError(f"Unsupported SVG path command {command!r}")

    if not vertices:
        return [], []
    return vertices, codes

def draw_world_svg(ax, svg_path="assets/world_silhouette_gray.svg"):
    """Draw the project's smooth gray SVG world silhouettes on a Matplotlib axis."""
    tree = ET.parse(svg_path)
    root = tree.getroot()
    namespace = "{http://www.w3.org/2000/svg}"
    paths = root.findall(f".//{namespace}path") or root.findall(".//path")
    if not paths:
        raise ValueError(f"No SVG path elements found in {svg_path}")
    viewbox = [float(v) for v in root.get("viewBox", "0 0 1000 500").split()]
    vx, vy, vw, vh = viewbox
    drawn = 0
    for element in paths:
        d = element.get("d")
        if not d:
            continue
        vertices, codes = parse_svg_path(d)
        if not vertices or codes[0] != MplPath.MOVETO:
            continue
        # Convert SVG's 0..1000 / 0..500 canvas into approximate lon/lat coordinates.
        geo_vertices = [((x - vx) / vw * 360.0 - 180.0, 90.0 - (y - vy) / vh * 180.0)
                        for x, y in vertices]
        ax.add_patch(PathPatch(MplPath(geo_vertices, codes), facecolor="#e3e6e9",
                               edgecolor="#c3cbd3", linewidth=0.65, alpha=0.9,
                               zorder=1, antialiased=True))
        drawn += 1
    if drawn == 0:
        raise ValueError(f"SVG parsed, but no drawable world paths were found in {svg_path}")
    ax.set_xlim(-175, 185)
    ax.set_ylim(-60, 85)
    return ax


## 4. Generate Chart

In [ ]:
fig = plt.figure(figsize=(18, 11), dpi=150, facecolor="#fbfaf7")
ax = fig.add_axes([0.04, 0.16, 0.92, 0.66])
ax.set_facecolor("#fbfaf7")
ax.axis("off")

draw_world_svg(ax)

max_v = max(v for v in values.values() if v is not None and v < 5000)
for country, v in values.items():
    if country not in pos:
        continue
    x, y = pos[country]
    if v is None:
        ax.scatter([x], [y], s=80, facecolors="none", edgecolors="#6b7280", linewidths=1.5, zorder=4)
        continue
    if country == "China":
        radius, color, edge = 6.5, "#f6e8d4", "#b96b26"
    else:
        radius = 1.6 + 7.5 * np.sqrt(v / max_v)
        color, edge = "#69c7d7", "#00a9c0"
    ax.add_patch(Circle((x, y), radius=radius, facecolor=color, edgecolor=edge,
                        linewidth=1.3, alpha=0.8, zorder=3))
    if v >= 200 and country != "China":
        ax.text(x, y, f"{int(v)}", ha="center", va="center", fontsize=9, color="#17333a", weight="bold", zorder=5)
        ax.text(x, y + 4.5, country, ha="center", va="center", fontsize=8, color="#17333a", weight="bold", zorder=6)
    elif country == "China":
        ax.text(x, y, "~15k", ha="center", va="center", fontsize=9, color="#7d4519", weight="bold", zorder=5)
        ax.text(x, y + 5, "China\n(cum.)", ha="center", va="center", fontsize=7.5, color="#7d4519", weight="bold", zorder=6)

fig.text(0.04, 0.92, "Quantum funding — 2026 snapshot", fontsize=22, weight="bold", color="#1a1a1a")
fig.text(0.04, 0.875, "Gray world silhouette. Bubble area scales with annual / annualized figure (USD millions).", fontsize=11, color="#444")
fig.text(0.70, 0.20, "CHINA\n≈ US$15B cumulative\nhigh uncertainty", ha="center", va="center", fontsize=8.5, color="#7d4519",
         bbox=dict(boxstyle="round,pad=0.5", facecolor="#f6e8d4", edgecolor="#b96b26"))
fig.text(0.04, 0.05, "US = FY2024 enacted. See REFERENCES.md.", fontsize=8, color="#555")
plt.show()

## Notes

- Parser fixed in v3.
- China is a cumulative estimate (not on the same scale).